# Sanity check: hybrid sampling (`reuse_frac`) in `optimize_LGD`

Runs the **2D cond 1D** LGD setup twice per seed — once with `reuse_frac=0.0` (current behavior, unchanged) and once with `reuse_frac=0.5` (half of the `nsamples` MMD-comparison batch at each step is carried over from the previous step instead of freshly sampled) — and compares final MMD, wall time, and the per-step gradient-norm trajectory, to confirm the reuse path is actually wired up and behaving as expected before running the full sweep.

In [ ]:
import os
# ============================================================
# CONFIG — same as Exp_2D_cond_1D.ipynb (fastest setup, good for a quick sanity check)
# ============================================================
EXPERIMENT_NAME   = "2D_cond_1D"
GLOBAL_SEED       = 42
FORCE_RETRAIN     = False

BASE_DIR = os.path.normpath(os.path.join(os.getcwd(), ".."))
PARAMS_DIR        = f"{BASE_DIR}/params"
CHECKPOINT_DIR    = f"{BASE_DIR}/checkpoints/{EXPERIMENT_NAME}"
RESULTS_DIR       = f"{BASE_DIR}/results/{EXPERIMENT_NAME}"

# Architecture — Diffusion models
NBLOCKS           = 3
NUNITS            = 128

# Architecture — Consistency Model (not used here, kept for parity with the loader)
NBLOCKS_CM        = 3
NUNITS_CM         = 128

# Training (only used if a checkpoint is missing)
NEPOCHS           = 20_000
BATCH_SIZE        = 1_024
NEPOCHS_CM        = 20_000
BATCH_SIZE_CM     = 1_024

# Diffusion
DIFFUSION_STEPS   = 100

# Optimization — hybrid-sampling sanity check
N_ATTEMP_OPTIM              = 10     # small: this is a sanity check, not the full sweep
NSAMPLES_IN_OPTIM_FOR_MMD   = 250
NUM_X_T_LGD                 = 3
REUSE_FRACS                 = [0.0, 0.5]

# GMM dimensions
CONDITION_ON      = 1   # dim(x)=1, dim(y)=1

In [ ]:
import os, sys

# ── point Python at simulations/src where all .py modules live ──
src_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "src")
src_path = os.path.normpath(src_path)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"src path on sys.path: {src_path}")

In [ ]:
import os, sys, time, json
import importlib
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial
from tqdm import trange

import Diffusion
import LossFunctions
import ConsistencyModels
import dist_utils
import Optimization
import experiment_utils
from ConsistencyModels import ConsistencyModeliCT
from LossFunctions import MMDLoss, RBF

for mod in [Diffusion, LossFunctions, ConsistencyModels,
            dist_utils, Optimization, experiment_utils]:
    importlib.reload(mod)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)
os.makedirs(PARAMS_DIR,     exist_ok=True)
print("Imports done.")

In [ ]:
env_info = experiment_utils.get_environment_info()
experiment_utils.print_environment_info(env_info)

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## GMM Parameters

In [ ]:
# ============================================================
# GMM PARAMETERS
# Priority:
#   1. Load from PARAMS_DIR  (shared across runs, committed to repo)
#   2. Load from RESULTS_DIR (fallback from a previous run)
#   3. Generate fresh and save to both dirs
# ============================================================
FORCE_REGENERATE_PARAMS = False

def _load_params():
    loaded = experiment_utils.load_gmm_params(PARAMS_DIR, EXPERIMENT_NAME)
    if loaded is not None:
        print(f"[GMM] Loaded from PARAMS_DIR: {PARAMS_DIR}")
        return loaded
    loaded = experiment_utils.load_gmm_params(RESULTS_DIR, EXPERIMENT_NAME)
    if loaded is not None:
        print(f"[GMM] Loaded from RESULTS_DIR: {RESULTS_DIR}")
        return loaded
    return None

loaded = None if FORCE_REGENERATE_PARAMS else _load_params()

if loaded is not None:
    mu_list, Sigma_list, alpha, mog_means, mog_variances, weights, x_star = loaded
    mu_list    = [mu.float() for mu in mu_list]
    Sigma_list = [cov.float() for cov in Sigma_list]
    alpha      = alpha.float()
else:
    print("[GMM] Generating fresh parameters...")
    experiment_utils.set_global_seed(GLOBAL_SEED)
    mu_list = [
        torch.tensor([-5,  5], dtype=torch.float64),
        torch.tensor([-5, -5], dtype=torch.float64),
        torch.tensor([ 5,  3], dtype=torch.float64),
        torch.tensor([ 5, -1], dtype=torch.float64),
        torch.tensor([ 0, -3], dtype=torch.float64),
        torch.tensor([-2,  4], dtype=torch.float64),
        torch.tensor([-2, -3], dtype=torch.float64),
        torch.tensor([ 1,  2], dtype=torch.float64),
        torch.tensor([-8,  1], dtype=torch.float64),
        torch.tensor([ 7,  5], dtype=torch.float64),
        torch.tensor([ 0, -5], dtype=torch.float64),
    ]
    Sigma_list = [
        torch.tensor([[0.5000, 0.1950],
                      [0.1950, 0.2000]], dtype=torch.float64)
    ] * len(mu_list)
    alpha = torch.tensor([1 / len(mu_list)] * len(mu_list), dtype=torch.float64)

    mu_list    = [mu.float() for mu in mu_list]
    Sigma_list = [cov.float() for cov in Sigma_list]
    alpha      = alpha.float()

    x_star = torch.tensor([-5])
    mu_temp, Sigma_temp = dist_utils.compute_conditionals(mu_list, Sigma_list, x_star)
    temp_alpha          = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_star)
    mog_means, mog_variances, weights = dist_utils.filter_and_normalize(
        mu_temp, Sigma_temp, temp_alpha, threshold=0.01
    )

    experiment_utils.save_gmm_params(
        mu_list, Sigma_list, alpha,
        mog_means, mog_variances, weights, x_star,
        PARAMS_DIR, EXPERIMENT_NAME
    )

print(f"x_star = {x_star}")
print(f"Number of conditional modes after filtering: {len(mog_means)}")

## Data

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)
X = dist_utils.generate_mog_samples(25_000, mu_list, Sigma_list, alpha).float().to(device)
xh_cpu = X.detach().cpu().numpy()
plt.scatter(xh_cpu[:, 0], xh_cpu[:, 1], alpha=0.6, s=20)
plt.title("Scatter Plot of P(X,Y)")
plt.xlabel("X"); plt.ylabel("Y"); plt.grid(True); plt.show()

## Load conditional diffusion model $P(Y|X=x)$
(This is the model whose `.sample()` calls are the target of `reuse_frac` — no CM model needed here, we're only checking the plain LGD path.)

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

X_train   = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)
nfeatures = X_train.shape[1]

data_generator_diff_cond = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha, kernel_func=None
)

model_cond = Diffusion.DiffusionModel(
    nfeatures=nfeatures, nblocks=NBLOCKS, nunits=NUNITS,
    condition=True, condition_on=CONDITION_ON,
    diffusion_steps=DIFFUSION_STEPS
)

_loaded_diff_cond = experiment_utils.load_checkpoint_with_hf_fallback(
    model_cond, "Diffusion_cond", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_diff_cond:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    model_cond.train_model(
        None, data_generator=data_generator_diff_cond,
        nepochs=NEPOCHS, batch_size=BATCH_SIZE,
        condition_on=CONDITION_ON
    )
    experiment_utils.save_model_checkpoint(
        model_cond, "Diffusion_cond", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

## Load unconditional diffusion model $P(X=x)$

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

data_generator_diff_uncond = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha,
    kernel_func=lambda X: X[:, :CONDITION_ON]
)

model_uncond = Diffusion.DiffusionModel(
    nfeatures=CONDITION_ON, nblocks=NBLOCKS, nunits=NUNITS,
    condition=False, diffusion_steps=DIFFUSION_STEPS
)

_loaded_diff_uncond = experiment_utils.load_checkpoint_with_hf_fallback(
    model_uncond, "Diffusion_uncond", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_diff_uncond:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    model_uncond.train_model(
        None, data_generator=data_generator_diff_uncond,
        nepochs=NEPOCHS, batch_size=BATCH_SIZE,
        condition_on=CONDITION_ON
    )
    experiment_utils.save_model_checkpoint(
        model_uncond, "Diffusion_uncond", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

## Run: `reuse_frac=0.0` vs `reuse_frac=0.5`

Same `N_ATTEMP_OPTIM` seeds for both settings (`experiment_utils.set_run_seed` reseeds identically per attempt, per `reuse_frac`), `return_history=True` so we get the per-step gradient norm, loss, and `n_reuse`/`n_new` back for each run.

In [ ]:
def run_sweep_point(reuse_frac):
    x_t_list, final_loss_list, times, histories = [], [], [], []
    for i in trange(N_ATTEMP_OPTIM, desc=f"reuse_frac={reuse_frac}"):
        run_seed = experiment_utils.set_run_seed(GLOBAL_SEED, i)

        start_time = time.time()
        best_x_t, _, final_loss, history = Optimization.optimize_LGD(
            model_uncond, model_cond, mog_means, mog_variances, weights,
            mu_list, Sigma_list, alpha,
            nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, loss="MMD", device=device,
            num_x_t=NUM_X_T_LGD, reuse_frac=reuse_frac, return_history=True,
        )
        end_time = time.time()

        x_t_list.append(best_x_t.reshape(-1, 1))
        final_loss_list.append(final_loss.item())
        times.append(end_time - start_time)
        histories.append(history)
        print(f"[reuse_frac={reuse_frac} | {i+1}/{N_ATTEMP_OPTIM}] seed={run_seed} "
              f"| final MMD: {final_loss.item():.6f} | time: {end_time - start_time:.2f}s")

    return {
        "x_t": x_t_list, "final_loss": final_loss_list,
        "times": times, "histories": histories,
    }

results_by_reuse = {}
for rf in REUSE_FRACS:
    results_by_reuse[rf] = run_sweep_point(rf)

## Sanity check 1 — is `reuse_frac` actually reusing samples?
`n_reuse` should be 0 on the very first optimization step (no buffer yet) and then equal to `round(reuse_frac * NSAMPLES_IN_OPTIM_FOR_MMD)` on every step after that, for the `reuse_frac=0.5` run — and 0 throughout for `reuse_frac=0.0`.

In [ ]:
for rf in REUSE_FRACS:
    h = results_by_reuse[rf]["histories"][0]   # first run's history
    n_reuse_after_first_step = [step["n_reuse"] for step in h[1:]]
    print(f"reuse_frac={rf}: step0 n_reuse={h[0]['n_reuse']}, "
          f"n_reuse thereafter (unique values)={sorted(set(n_reuse_after_first_step))}, "
          f"expected={round(rf * NSAMPLES_IN_OPTIM_FOR_MMD)}")

## Sanity check 2 — final MMD & wall time

In [ ]:
rows = []
for rf in REUSE_FRACS:
    r = results_by_reuse[rf]
    rows.append({
        "reuse_frac":   rf,
        "MMD mean":     np.mean(r["final_loss"]),
        "MMD std":      np.std(r["final_loss"]),
        "Time mean (s)":np.mean(r["times"]),
        "Time std (s)": np.std(r["times"]),
    })
summary_df = pd.DataFrame(rows)
summary_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].boxplot([results_by_reuse[rf]["final_loss"] for rf in REUSE_FRACS],
                labels=[str(rf) for rf in REUSE_FRACS])
axes[0].set_xlabel("reuse_frac"); axes[0].set_ylabel("final MMD")
axes[0].set_title("Final MMD distribution"); axes[0].grid(True, alpha=0.3)

axes[1].boxplot([results_by_reuse[rf]["times"] for rf in REUSE_FRACS],
                labels=[str(rf) for rf in REUSE_FRACS])
axes[1].set_xlabel("reuse_frac"); axes[1].set_ylabel("wall time (s)")
axes[1].set_title("Per-run wall time"); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Sanity check 3 — gradient-norm trajectory over the reverse-diffusion steps
This is the key mechanism check: with `reuse_frac=0.5`, gradient at each step flows only through the freshly-generated half of the MMD batch (the reused half is detached), so we expect a *noisier / not-necessarily-smaller* grad-norm curve with the same rough shape as `reuse_frac=0.0` — not a curve that's silently zero or identical to the baseline (which would mean the reuse path isn't wired in).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for rf in REUSE_FRACS:
    histories = results_by_reuse[rf]["histories"]
    # align by step index t (all runs share the same DIFFUSION_STEPS schedule)
    steps = [step["t"] for step in histories[0]]
    grad_matrix = np.array([[step["grad_norm"] for step in h] for h in histories])  # (n_runs, n_steps)
    mean_grad = grad_matrix.mean(axis=0)
    std_grad  = grad_matrix.std(axis=0)

    ax.plot(steps, mean_grad, label=f"reuse_frac={rf}")
    ax.fill_between(steps, mean_grad - std_grad, mean_grad + std_grad, alpha=0.2)

ax.invert_xaxis()  # t goes from diffusion_steps-1 down to 1
ax.set_xlabel("diffusion step t"); ax.set_ylabel("||grad x_t|| (mean ± std over runs)")
ax.set_title("Gradient-norm trajectory: reuse_frac=0.0 vs 0.5")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Sanity check 4 — `reuse_frac=0.0` reproduces the pre-change code path
With `reuse_frac=0.0`, `n_reuse` is always 0, so every step generates the full `nsamples` batch fresh — identical to `optimize_LGD` before this change. This just re-affirms that from the collected history (all `n_new == NSAMPLES_IN_OPTIM_FOR_MMD`).

In [ ]:
h0 = results_by_reuse[0.0]["histories"][0]
all_full_batch = all(step["n_new"] == NSAMPLES_IN_OPTIM_FOR_MMD for step in h0)
print(f"reuse_frac=0.0 always uses a full fresh batch every step: {all_full_batch}")

## Save results

In [ ]:
def to_python(val):
    if isinstance(val, torch.Tensor):
        return val.detach().cpu().tolist()
    if isinstance(val, np.ndarray):
        return val.tolist()
    if hasattr(val, "item"):
        return val.item()
    return val

out = {
    "experiment": EXPERIMENT_NAME,
    "seed": GLOBAL_SEED,
    "environment": env_info,
    "meta": {
        "n_attemp_optim": N_ATTEMP_OPTIM,
        "nsamples_in_optim_for_mmd": NSAMPLES_IN_OPTIM_FOR_MMD,
        "num_x_t": NUM_X_T_LGD,
        "reuse_fracs": REUSE_FRACS,
    },
    "by_reuse_frac": {
        str(rf): {
            "final_loss": results_by_reuse[rf]["final_loss"],
            "times": results_by_reuse[rf]["times"],
            "histories": results_by_reuse[rf]["histories"],
        }
        for rf in REUSE_FRACS
    },
}

path = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_hybrid_sanity_check_seed{GLOBAL_SEED}.json")
with open(path, "w") as f:
    json.dump(out, f, indent=2)
print(f"Results saved to {path}")